### pyAPES MLM test run for Degerö Stormyr (SE-Deg), pristine peatland ICOS-site in Northern Sweden


v. 24.04.2026 / Samuli tested after code modifications

In [1]:
# setting pyAPES_main to path
import sys
import os
from dotenv import load_dotenv

load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')

sys.path.append(pyAPES_main_folder)

### Import modules

In [2]:
# function to read forcing data. See 'forcing/forcing_info.txt' for model forcing variable names and units!
from pyAPES.utils.iotools import read_forcing

# import the multi-layer model (mlm) driver
from pyAPES.pyAPES_MLM import driver

# python packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib

eps = 1e-16

%matplotlib qt

In [4]:
# import parameter dictionaries
from pyAPES.parameters.mlm_parameters_SE_Deg import gpara, cpara, spara # model configuration, canopy parameters, soil parameters

In [5]:
forcing = read_forcing(
    forcing_file=gpara['forc_filename'],
    start_time=gpara['start_time'],
    end_time=gpara['end_time'],
    dt=gpara['dt']
)

params = {
    'general': gpara,   # model configuration
    'canopy': cpara,    # planttype, micromet, canopy, bottomlayer parameters
    'soil': spara,      # soil heat and water flow parameters
    'forcing': forcing  # forging data
}

In [6]:
resultfile, Model = driver(parameters=params,
                           create_ncf=True,
                           result_file= 'SEDeg_test.nc'
                          )

# alternative (for testing over short periods) is to return resuts in dictionary 'results'
# results, Model = driver(parameters=params,
#                         create_ncf=False,
#                         result_file=None
#                          )

INFO pyAPES.pyAPES_MLM driver pyAPES_MLM simulation started. Number of simulations: 1
INFO pyAPES.soil.water __init__ Water balance in soil solved using: EQUILIBRIUM & HOOGHOUDT
INFO pyAPES.soil.heat __init__ Soil heat balance solved.
INFO pyAPES.canopy.mlm_canopy __init__ Eflow: True, WMA: False, Ebal: True
INFO pyAPES.microclimate.radiation __init__ Shortwave radiation model: ZHAOQUALLS
INFO pyAPES.microclimate.radiation __init__ Longwave radiation model: ZHAOQUALLS
INFO pyAPES.canopy.forestfloor __init__ Forestfloor has 1 bottomlayer types
INFO pyAPES.utils.iotools save_parameters_yaml Parameters saved to: c:\Repositories\pyAPES_main\inputs\SEDeg_test.yml
INFO pyAPES.pyAPES_MLM driver Running simulation number (start time 2026-04-23 16:32): 0
INFO pyAPES.pyAPES_MLM run Running simulation 0


0%.. 10%.. 20%.. 30%.. 40%.. 50%.. 60%.. 70%.. 80%.. 90%.. 

INFO pyAPES.pyAPES_MLM run Finished simulation 0, running time 44.19 seconds
INFO pyAPES.pyAPES_MLM driver Running time 44.20 seconds


100%


INFO pyAPES.pyAPES_MLM driver Ready! Results are in: c:\\Repositories\\pyAPES_main/results/SEDeg_test.nc


### Read model results using xarray
- use function *pyAPES.utils.utils.read_results* to open NetCDF4 file storing the results. It uses xarray, see https://docs.xarray.dev/en/stable/

- compare some model results against fluxes and state variables from US-PRR (FluxNet2015 data subset compiled in *Ex1_creating_model_forcing.ipynb*)

In [7]:
from pyAPES.utils.iotools import read_results, read_data

#resultfile = pathlib.Path(fr'{pyAPES_main_folder}/results/USPrr_test.nc')
datafile = pathlib.Path(fr'{pyAPES_main_folder}/forcing/Degero/Degero_EC_2014_2016_ICOS.csv')

# read simulation restuls to xarray dataset
results = read_results(resultfile)

# read observations from US-Prr
data = read_data(datafile, start_time=gpara['start_time'], end_time=gpara['end_time'])

c:\\Repositories\\pyAPES_main/results/SEDeg_test.nc


KeyError: "None of [Index(['year', 'month', 'day', 'hour', 'minute'], dtype='object')] are in the [columns]"

### Print resuts metadata:

Output variables are those defined at pyAPES.parameters.mlm_outputs'. Each variable contains one or several of following dimensions:

- date --> time dimension
- simulation --> simulation number. If there is only one, remember that it has index 0!
- canopy --> canopy and air layers. Index is 0 at ground and increases upwards
- planttype --> planttypes defined in mlm_parameters. Index follows the order in dict cpara['planttypes']) 
- soil -> soil layers. Index 0 is at the top and increases downwards (nr soil layers, 0 at top)
- groundtype --> groundtypes at forestfloor. Defined in mlm_parameters. Index follows the order in dict: cpara['forestfloor']['bottom_layer_types']


In [ ]:
print(results)

# print list of all variables:
vars = list(results.data_vars)
for v in vars:
    print(v)

### Plot some model results and compare with observations at US-Prr

- use FluxNet2015 data subset created in *Ex1_creating_model_forcing.ipynb*

In [ ]:
sim = 0  # in this demo, we have only one simulation (i.e. only one parameter set was used)

# grid variables for plotting
t = results.date  # time
zc = results.canopy_z  # height above ground [m]
zs = results.soil_z  # depth of soil; shown negative [m]

### Soil temperature and moisture
- computed using *pyAPES.soil* submodels 'Water' and 'Heat'
- compare with measurements at similar depths (to be done - now compare with Ts and SWC at 5cm depth)

In [ ]:

var = ['soil_temperature', 'soil_volumetric_liquid_water_content', 'soil_volumetric_ice_content']

lyrs = [0, 6, 25, 35] # layers
#depths = np.array2string(np.asarray(zs[lyrs]), precision=1, separator=', ')
depths = ['{:.2f} m'.format(k) for k in zs[lyrs]]

fig, ax = plt.subplots(len(var), 1, figsize=(10,7))

k = 0
ax[0].plot(t, data['TS_F_MDS_1'], 'ko', markersize=3, alpha=0.1)
ax[1].plot(t, 1e-2 * data['SWC_F_MDS_2'], 'ko', markersize=3, alpha=0.1)
for v in var:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)
    k += 1

# # vertical profile at last timestep
# fig, ax = plt.subplots(1, 2) #figsize=(10,5))

# k = 0
# for v in var:
#     ax[k].plot(results[v][-1,sim,:], zs)
#     ax[k].set_xlabel(results[v].attrs['units'])
#     ax[0].set_ylabel('depth (m)')
#     k += 1

### Ecosystem-scale fluxes

- ecosystem - atm. fluxes represent the integrated sinks / sources in soil (soil-module), forestfloor (bottomlayer-module) and vegetation (planttype&canopy -modules)
- comparable to ecosystem - atmosphere exchange
- affected by current model forcing and sub-model instance state (e.g. Planttype and Canopy LAI, phenology, soil temperature, moisture etc.)


In [ ]:

# var = ['forcing_air_temperature', 'forcing_par', 'canopy_NEE', 'canopy_GPP', 
#        'canopy_Reco', 'canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']

var = ['canopy_NEE', 'canopy_GPP', 'canopy_Reco']
var2 = ['canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']


# # temperature & par
# ax[0].plot(t, results[var[0]][:,sim], 'k-', label=var[0])
# ax[0].set_ylabel(results[var[0]].attrs['units'])
# ax[0].tick_params(axis='x', labelrotation = 20)
# axb = ax[0].twinx()

# axb.plot(t, results[var[1]][:,sim], 'r-', label=var[1]) # Par
# axb.set_ylabel(results[var[1]].attrs['units'])

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['NEE_VUT_REF'], 'k.-', alpha=0.3)
ax[1].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3)
ax[2].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3)
n = 0
for v in var:
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('umol m-2 s-1')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n +=1

fig, ax = plt.subplots(5, 1, figsize=(8,12), sharex=True)
# energy fluxes
ax[1].plot(t, data['NETRAD'], 'k.-', alpha=0.3)
ax[2].plot(t, data['H_F_MDS'], 'k.-', alpha=0.3)
ax[3].plot(t, data['LE_F_MDS'], 'k.-', alpha=0.3)
ax[4].plot(t, data['G_F_MDS'], 'k.-', alpha=0.3)

n = 1
for v in var2:
    ax[0].plot(t, results[v][:,sim], label=v)
    ax[n].plot(t, results[v][:,sim], label=v)

    ax[n].set_ylabel('W m-2')
    ax[n].tick_params(axis='x', labelrotation = 20)
    ax[n].legend(fontsize=8)
    n += 1


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(8,12), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['GPP_NT_VUT_REF'], 'k.-', alpha=0.3); ax[0].set_ylabel('GPP (umolm-2s-1)')
ax[0].plot(t, results['canopy_GPP'][:,sim], label='GPP')

ax[1].plot(t, data['RECO_NT_VUT_REF'], 'k.-', alpha=0.3); ax[1].set_ylabel('RECO (umolm-2s-1)')
ax[1].plot(t, results['canopy_Reco'][:,sim], label='GPP')
ax[2].plot(t, data['TS_F_MDS_1'], 'k.-', alpha=0.3); ax[2].set_ylabel('T (degC)')
ax[3].plot(t, 1e-2*data['SWC_F_MDS_2'], 'k.-', alpha=0.3); ax[3].set_ylabel('SWC (m3m-3)')

ax[2].plot(t, results['soil_temperature'][:,sim,lyrs], label=depths)
ax[2].legend(fontsize=8)

var3 = ['soil_volumetric_liquid_water_content']
k = 3
for v in var3:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)



### Ecosystem radiation balance

- net radiation (Rnet) is the sum of net shortwave (SWnet = incoming - reflected) and net longwave (LWnet = incoming - emitted) radiation at canopy top
- computed via models in pyAPES.microclimate.radiation, called iteratively from pyAPES.canopy.mlm_canopy to account for canopy structure and leaf & forest floor temperature
- ecosystem albedo can be computed from radiation profiles at uppermost grid point

In [ ]:
# net radiation components at canopy top
var = ['canopy_Rnet','canopy_SWnet', 'canopy_LWnet']
profs = ['canopy_par_down', 'canopy_par_up', 'canopy_nir_down','canopy_nir_up','canopy_lw_down','canopy_lw_up']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

for v in var:
    ax[0].plot(t, results[v][:, sim], label=v)
ax[0].set_ylabel('W m-2')
ax[0].tick_params(axis='x', labelrotation = 20)
ax[0].legend(fontsize=8)    

# lets plot also partitioning of canopy_SWnet and canopy_LWnet.
# -1 is the index of uppermost gridpoint

for v in ['canopy_par_down', 'canopy_nir_down','canopy_lw_down']: # downward
    ax[1].plot(t, results[v][:,sim,-1], '-', label=v)
for v in ['canopy_par_up', 'canopy_nir_up','canopy_lw_up']: # upward
    ax[1].plot(t, results[v][:,sim,-1], '--', label=v)
ax[1].set_ylabel('W m-2')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# canopy albedo
eps = 1e-16

# fraction of par on total SW
f_par = results['canopy_par_down'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + results['canopy_nir_down'][:,sim,-1] + eps)

alb_par = results['canopy_par_up'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + eps)
alb_nir = results['canopy_nir_up'][:,sim,-1] / (results['canopy_nir_down'][:,sim,-1] + eps)
alb_sw = f_par * alb_par + (1 - f_par) * alb_nir
alb_sw = np.maximum(0, np.minimum(1.0, alb_sw))

ax[2].plot(t, alb_sw, '-', label='SW albedo')
ax[2].plot(t, alb_par, '-', label='Par albedo')
ax[2].plot(t, alb_nir, '-', label='Nir albedo')

ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)

### Microclimatic gradients within the canopy

- sub-models pyAPES.microclimate.Micromet & pyAPES.radiation.Radiation
- mean wind speed & friction velocity (momentum flux) profiles depend on momentum absorption in the canopy, which is proportional to leaf-area density (*lad*) profile. Canopy *lad* is sum of *lad* of each PlantType instance
- SW absorption depends on *lad* and leaf optical properties
- air-space scalar profiles (*T*, *H2O* & *CO2*) in steady-state with respective sink/source profile using 1st-order closure (K-theory)

Let's compute average profiles for daytime hours 10:00 -- 16:00 under conditions when canopy is dry


In [ ]:
results[v]

In [ ]:
par = results['forcing_par'][:,sim] # use PAR as criteria for night / day
ix = np.where((results.date.dt.hour>=10) & (results.date.dt.hour<=16) & (results['canopy_interception_storage'] <= eps))

rad = ['canopy_par']
mmet = ['canopy_wind_speed', 'canopy_friction_velocity', 'canopy_temperature', 'canopy_co2', 'canopy_h2o']

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(10,10))
fig.subplots_adjust(wspace=0.4, hspace=0.4)
axs = axs.flatten()
n = 1
for v,ax in zip(mmet,axs):
    x = results[v][:, sim, :] - results[v][:, sim, -1] # s - s_ref
    xm = np.mean(x, axis=0)
    xmn = np.mean(x.loc[par < 20, :], axis=0) # night
    xmd = np.mean(x.loc[par > 200, :], axis=0) # day
    ax.plot(xm, zc, 'k', alpha=0.1, label='day&night')
    ax.plot(xmd, zc, 'r', label='day')
    ax.plot(xmn, zc, 'b', label='night')
    ax.set_xlabel(results[v].attrs['units'])
    ax.set_ylabel('Canopy height [m]')
    n += 1

axs[-2].legend()
axs[-1].remove()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pyAPES.soil.heat import sinusoidal_soil_temperature

# --- grid ---
doys = np.arange(1, 366)
dz = np.array([0.05] * 100)
z = dz / 2 - np.cumsum(dz)          # node elevations, negative downward [m]

# --- model parameters ---
T_mean = 8.0
T_amplitude = 10.0
thermal_diffusivity = 5e-7

# --- compute T(z, doy) ---
T_grid = np.array([
    sinusoidal_soil_temperature(z, doy=d, T_mean=T_mean, T_amplitude=T_amplitude, thermal_diffusivity=thermal_diffusivity,)
    for d in doys
]).T  # shape (n_depths, n_doys)

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))

levels = np.arange(-6, 18, 1)
cf = ax.contourf(doys, z, T_grid, levels=levels, cmap='RdBu_r', extend='both')
cs = ax.contour(doys, z, T_grid, levels=levels, colors='k', linewidths=0.5, alpha=0.5)
ax.clabel(cs, levels=levels[::2], fmt='%d°C', fontsize=7, inline=True)

cbar = fig.colorbar(cf, ax=ax, label='Soil temperature (°C)')

ax.set_xlabel('Day of year')
ax.set_ylabel('Depth (m)')
ax.set_title(f'Sinusoidal soil heat-wave model  |  $\\bar{{T}}$={T_mean}°C, $A_0$={T_amplitude}°C')
ax.xaxis.set_major_locator(ticker.MultipleLocator(30))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
ax.set_xlim(1, 365)
ax.set_ylim(z[-1], 0)

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.dates as mdates

tmp = results['soil_volumetric_water_content'][:, sim, :].values.T  # shape: (soil, date)
z = zs.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, tmp, cmap='YlOrRd', shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='m3m-3')
ax.set_ylabel('depth (m)')
ax.set_xlabel('Date')
ax.set_title('vol water content (m3m-3)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

tmp = results['soil_temperature'][:, sim, :].values.T  # shape: (soil, date)
z = zs.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, tmp, cmap='YlOrRd', shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='m3m-3')
ax.set_ylabel('depth (m)')
ax.set_xlabel('Date')
ax.set_title('soil temperature (degC)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:

tmp = results['soil_heat_flux'][:, sim, :].values.T  # shape: (soil, date)
z = zs.values
T = t.values

vmax = np.nanpercentile(np.abs(tmp), 98)
levels = np.linspace(-vmax, vmax, 25)

fig, ax = plt.subplots(figsize=(12, 5))
cf = ax.contourf(T, z, tmp, levels=levels, cmap='RdBu_r', extend='both')
cs = ax.contour(T, z, tmp, levels=levels, colors='k', linewidths=0.5, alpha=0.5)
ax.clabel(cs, levels=levels[::4], fmt='%.1f', fontsize=7, inline=True)

cbar = fig.colorbar(cf, ax=ax, label=results['soil_heat_flux'].attrs.get('units', 'W m-2'))
ax.set_ylabel('Depth (m)')
ax.set_xlabel('Date')
ax.set_title('Soil heat flux')
ax.set_ylim(z[-1], z[0])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
results.close()